# **El problema de negocio**

El proyecto busca desarrollar una solución que permita analizar el consumo eléctrico de hogares y pequeños establecimientos para clasificar el perfil energético de los usuarios en función de sus patrones de consumo y las características de su entorno, con el objetivo de promover la eficiencia energética y generar recomendaciones de ahorro.

**¿Qué factores se asocian de manera significativa con la clasificación de un usuario como eficiente, moderado o ineficiente?**

# **Fase 1 — Configuración inicial**

In [ ]:
!pip install pandas numpy requests plotly scikit-learn scipy statsmodels

In [ ]:
import pandas as pd
import numpy as np
import requests
import plotly.express as px

In [ ]:
url = "https://raw.githubusercontent.com/No-Country-simulation/G9-LATAM-TEAM-09/refs/heads/develop/data-science/data/database_beta.json"
respuesta = requests.get(url)
datos_json = respuesta.json()
df_energIA = pd.json_normalize(datos_json)

In [ ]:
df_energIA.head()

,hogar_id,tipo_inmueble,metros_cuadrados,antiguedad_vivienda,zona_fria,calidad_aislamiento,fuente_calefaccion,fuente_agua_caliente,consumo_kwh,uso_horario_pico,horas_alto_consumo,cantidad_equipos,categoria
0,Hogar_0001,Departamento,1269,61,No,Muy Baja,Solar,Electricidad,364.0,Si,14,19,Moderado
1,Hogar_0002,Pyme,928,16,Si,Media,Electricidad,Otros,730.5,No,5,92,Ineficiente
2,Hogar_0003,Comercio,188,65,No,Media,Otros,Otros,299.3,Si,7,39,Moderado
3,Hogar_0004,Departamento,2000,5,Si,Muy Baja,Electricidad,Solar,543.4,Si,23,85,Ineficiente
4,Hogar_0005,Casa,1269,57,No,Muy Alta,Electricidad,Electricidad,433.9,Si,13,90,Moderado


# **Fase 2 — Información general**

In [ ]:
df_energIA.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   hogar_id              2000 non-null   object 
 1   tipo_inmueble         2000 non-null   object 
 2   metros_cuadrados      2000 non-null   int64  
 3   antiguedad_vivienda   2000 non-null   int64  
 4   zona_fria             2000 non-null   object 
 5   calidad_aislamiento   2000 non-null   object 
 6   fuente_calefaccion    2000 non-null   object 
 7   fuente_agua_caliente  2000 non-null   object 
 8   consumo_kwh           2000 non-null   float64
 9   uso_horario_pico      2000 non-null   object 
 10  horas_alto_consumo    2000 non-null   int64  
 11  cantidad_equipos      2000 non-null   int64  
 12  categoria             2000 non-null   object 
dtypes: float64(1), int64(4), object(8)
memory usage: 203.3+ KB


La información recolectada se encuentra almacenada en un archivo JSON (`database_beta.json`), compuesto por 2.000 registros únicos (identificados del 0 al 1999) y 13 columnas.

Cada registro incluye un identificador único (columna 0), 11 variables relacionadas con los patrones de consumo y las características del entorno (columnas 1 a 11), y una variable objetivo correspondiente a la categoría del perfil energético (columna 12).


# **Fase 3 — Revisión tipos de datos**

In [ ]:
df_energIA.dtypes

,0
hogar_id,object
tipo_inmueble,object
metros_cuadrados,int64
antiguedad_vivienda,int64
zona_fria,object
calidad_aislamiento,object
fuente_calefaccion,object
fuente_agua_caliente,object
consumo_kwh,float64
uso_horario_pico,object


In [ ]:
col_object_a_bool = [
    "zona_fria",
    "uso_horario_pico"
]

for col in col_object_a_bool:
    df_energIA[col] = df_energIA[col].map({
        "Si": True,
        "No": False
    }).astype("bool")

In [ ]:
df_energIA.dtypes

,0
hogar_id,object
tipo_inmueble,object
metros_cuadrados,int64
antiguedad_vivienda,int64
zona_fria,bool
calidad_aislamiento,object
fuente_calefaccion,object
fuente_agua_caliente,object
consumo_kwh,float64
uso_horario_pico,bool


# **Fase 4 — Revisión valores**

In [ ]:
df_energIA.isnull().sum()

,0
hogar_id,0
tipo_inmueble,0
metros_cuadrados,0
antiguedad_vivienda,0
zona_fria,0
calidad_aislamiento,0
fuente_calefaccion,0
fuente_agua_caliente,0
consumo_kwh,0
uso_horario_pico,0


In [ ]:
df_energIA.duplicated().sum()

np.int64(0)

Se verificó que la base de datos no presenta valores nulos ni registros duplicados.

# **Fase 5 — Estadística descriptiva**

In [ ]:
# Variables categóricas
df_energIA.describe(exclude=('int64','float64'))

,hogar_id,tipo_inmueble,zona_fria,calidad_aislamiento,fuente_calefaccion,fuente_agua_caliente,uso_horario_pico,categoria
count,2000,2000,2000,2000,2000,2000,2000,2000
unique,2000,4,2,5,3,3,2,3
top,Hogar_1984,Casa,False,Media,Electricidad,Electricidad,True,Moderado
freq,1,705,1203,676,901,901,1221,1331


In [ ]:
# Variables numéricas
df_energIA.describe().map('{:.2f}'.format)

,metros_cuadrados,antiguedad_vivienda,consumo_kwh,horas_alto_consumo,cantidad_equipos
count,2000.00,2000.00,2000.00,2000.00,2000.00
mean,1047.92,76.57,502.13,12.07,50.63
std,575.89,43.39,287.26,7.18,28.78
min,26.00,0.00,1.10,0.00,1.00
25%,541.75,38.75,253.70,6.00,27.00
50%,1062.00,78.00,506.45,12.00,50.00
75%,1574.00,114.00,747.70,18.00,76.00
max,2000.00,150.00,999.90,24.00,100.00


# **Fase 6 — Análisis variables categóricas**

In [ ]:
# Función gráfico de barras
def barras(df, variable_x, variable_y, titulo, etiqueta_x, etiqueta_y, leyenda):
    fig = px.bar(
        df,
        x=variable_x,
        y=variable_y,
        title=titulo,
        color=variable_x,
        color_discrete_sequence=px.colors.sequential.Rainbow_r
    )

    fig.update_layout(
        title_font_size=24,
        xaxis_title=etiqueta_x,
        yaxis_title=etiqueta_y,
        hoverlabel=dict(
            bgcolor="white",
        ),
        legend=dict(
            title_text=leyenda,
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.show()

In [ ]:
# Variable tipo_inmueble
conteo_inmuebles = df_energIA["tipo_inmueble"].value_counts().reset_index()
conteo_inmuebles.columns = ["tipo_inmueble", "cantidad"]

barras(conteo_inmuebles,
       'tipo_inmueble',
       'cantidad',
       'Distribución variable tipo_inmueble',
       'Tipo de inmueble',
       'Cantidad',
       'Tipo de inmueble')

In [ ]:
# Variable tipo_inmueble
orden = ["Muy Alta", "Alta", "Media", "Baja", "Muy Baja"]
conteo_aislamiento = df_energIA["calidad_aislamiento"].value_counts().reindex(orden).reset_index()
conteo_aislamiento.columns = ["calidad_aislamiento", "cantidad"]

barras(conteo_aislamiento,
       'calidad_aislamiento',
       'cantidad',
       'Distribución variable calidad_aislamiento',
       'Calidad de aislamiento',
       'Cantidad',
       'Calidad')

In [ ]:
# Variable fuente_calefaccion
conteo_calefaccion = df_energIA["fuente_calefaccion"].value_counts().reset_index()
conteo_calefaccion.columns = ["fuente_calefaccion", "cantidad"]

barras(conteo_calefaccion,
       'fuente_calefaccion',
       'cantidad',
       'Distribución variable fuente_calefaccion',
       'Fuente principal de calefacción',
       'Cantidad',
       'Fuentes')

In [ ]:
# Variable fuente_agua_caliente
conteo_agua = df_energIA["fuente_agua_caliente"].value_counts().reset_index()
conteo_agua.columns = ["fuente_agua_caliente", "cantidad"]

barras(conteo_agua,
       'fuente_agua_caliente',
       'cantidad',
       'Distribución variable fuente_agua_caliente',
       'Fuente principal de calefacción de agua caliente',
       'Cantidad',
       'Fuentes')

Observaciones:

1. `tipo_inmueble`: 4 valores (Casa, Departamento, Comercio y Pyme). Predominan los inmuebles residenciales (65,05% del total), correspondientes a Casa y Departamento, mientras que los inmuebles de uso comercial y productivo (Comercio y Pyme) presentan una menor representación.

2. `calidad_aislamiento`: 5 valores (Muy Alta, Alta, Media, Baja y Muy Baja). La categoría Media concentra la mayor cantidad de registros. Además, las categorías de menor calidad de aislamiento (Baja y Muy Baja) superan en frecuencia a las de mayor calidad (Alta y Muy Alta), lo que indica que, en términos generales, el nivel de aislamiento tiende a ser intermedio con una inclinación hacia valores bajos.

3. `fuente_calefaccion`: 3 valores (Electricidad, Solar y Otros). La categoría Electricidad es la más frecuente de manera individual. Sin embargo, al agrupar las categorías Solar y Otros, estas representan el 54,95% de los registros, superando a Electricidad en conjunto.

4. `fuente_agua_caliente`: 3 valores (Electricidad, Solar y Otros). Se observa un comportamiento similar al de `fuente_calefaccion`, con una mayor participación de Electricidad. En comparación con la variable anterior, hubo una disminución en Solar, donde registran 27 casos transfiriéndose a Otros.






# **Fase 7 — Análisis variables numéricas**

In [ ]:
# Función diagrama de caja
def boxplot(df, variable_x, titulo, etiqueta_x):
    fig = px.box(
        df,
        x=variable_x,
        points="outliers",
        title=titulo,
        color_discrete_sequence=px.colors.sequential.Rainbow
    )

    fig.update_layout(
        title_font_size=24,
        xaxis_title=etiqueta_x,
        hoverlabel=dict(
            bgcolor="white",
        )
    )

    fig.show()

In [ ]:
# Función gráfico circular
def piechart(df, variable_x, variable_y, titulo, leyenda):
    fig = px.pie(
        df,
        names=variable_x,
        values=variable_y,
        title=titulo,
        color_discrete_sequence=px.colors.sequential.Rainbow_r
    )

    fig.update_layout(
        title_font_size=24,
        hoverlabel=dict(
            bgcolor="white",
        ),
        legend=dict(
            title_text=leyenda,
            orientation="v",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    fig.update_traces(
        textfont_color='white',
        textfont_size=14)

    fig.show()

In [ ]:
# Variable metros_cuadrados
boxplot(
    df_energIA,
    "metros_cuadrados",
    "Distribución variable metros_cuadrados",
    "Metros cuadrados"
    )

In [ ]:
# Variable antiguedad_vivienda
boxplot(
    df_energIA,
    "antiguedad_vivienda",
    "Distribución de la variable antiguedad_vivienda",
    "Antiguedad de la vivienda en años"
)

In [ ]:
# Variable zona_fria
conteo_zona = df_energIA["zona_fria"].value_counts().reset_index()
conteo_zona.columns = ["zona_fria", "cantidad"]

piechart(
    conteo_zona,
    "zona_fria",
    "cantidad",
    "Distribución variable zona_fria",
    "Zona Fría")

In [ ]:
# Variable consumo_kwh
boxplot(
    df_energIA,
    "consumo_kwh",
    "Distribución variable consumo_kwh",
    "Consumo energético en KWh"
)

In [ ]:
# Variable uso_horario_pico
conteo_horario = df_energIA["uso_horario_pico"].value_counts().reset_index()
conteo_horario.columns = ["uso_horario_pico", "cantidad"]
conteo_horario["uso_horario_pico"] = conteo_horario["uso_horario_pico"].map({
    True: "Sí",
    False: "No"
})

piechart(
    conteo_horario,
    "uso_horario_pico",
    "cantidad",
    "Distribución variable uso_horario_pico",
    " ")

In [ ]:
# Variable horas_alto_consumo
boxplot(
    df_energIA,
    "horas_alto_consumo",
    "Distribución variable horas_alto_consumo",
    "Horas de consumo alto"
)

In [ ]:
# Variable cantidad_equipos
boxplot(
    df_energIA,
    "cantidad_equipos",
    "Distribución variable cantidad_equipos",
    "Cantidad de equipos"
)

Observaciones:

1. `metros_cuadrados`: (26 a 2000). El 50% de los registros se concentra entre 541.5 y 1574 m², con una mediana de 1062 m². La distribución presenta una asimetría negativa, indicando una mayor dispersión hacia los inmuebles de menor superficie.
2. `antiguedad_vivienda`: (0 a 150). El 50% de los registros se concentra entre 38.5 y 114 años, con una mediana de 78 años. La distribución presenta una leve asimetría negativa, lo que indica una ligera mayor dispersión hacia inmuebles de menor antigüedad.
3. `zona_fria`: 2 valores (Sí, No). La distribución de los registros se encuentra inclinada hacia "No". Un total de 796 inmuebles (39,9%) se ubican en zona fría, mientras que 1204 (60,2%) corresponden a inmuebles fuera de dicha zona.
4. `consumo_kwh`: (1.0 a 1000 kWh). El 50% de los registros se concentra entre 253.7 y 747.7 kWh, con una mediana de 506.45 kWh. La distribución presenta una leve asimetría negativa.
5. `uso_horario_pico`: 2 valores (Sí, No). Se observa una diferencia del 22,1% entre los usuarios que registran consumo en horario pico y aquellos que no. El 61,1% de los registros (1222 casos) corresponden a la categoría "No", mientras que el 39% (778 casos) pertenecen a la categoría "Si".
6. `horas_alto_consumo`: (0 a 24). El 50% central de los registros se concentra entre 6 y 18 horas diarias, con una mediana de 12 horas. La distribución presenta una leve asimetría positiva, indicando una ligera mayor dispersión hacia valores más altos.
7. `cantidad_equipos`: (1 a 100). El 50% central de los registros se concentra entre 27 y 76 equipos, con una mediana de 50 equipos. La distribución presenta una leve asimetría positiva.

# **Fase 8 — Análisis y corrección de outliers**

## **Variables numéricas**

In [ ]:
# Función outliers variables numéricas
def detectar_outliers(df, variable):
  q1 = df[variable].quantile(0.25)
  q3 = df[variable].quantile(0.75)

  iqr = q3-q1

  li_inf = q1 - 1.5*iqr
  li_sup = q3 + 1.5*iqr

  outliers = df[(df[variable] < li_inf) | (df[variable] > li_sup)]
  cantidad_outliers = outliers.shape[0]

  print(f'Cantidad de outliers para la variable {variable}: ' + str(cantidad_outliers))
  if (cantidad_outliers != 0):
    return outliers

In [ ]:
# Outliers para la variable metros_cuadrados
detectar_outliers(df_energIA, "metros_cuadrados")

Cantidad de outliers para la variable metros_cuadrados: 0


In [ ]:
# Outliers para la variable antiguedad_vivienda
detectar_outliers(df_energIA, "antiguedad_vivienda")

Cantidad de outliers para la variable antiguedad_vivienda: 0


In [ ]:
# Outliers para la variable consumo_kwh
detectar_outliers(df_energIA, "consumo_kwh")

Cantidad de outliers para la variable consumo_kwh: 0


In [ ]:
# Outliers para la variable horas_alto_consumo
detectar_outliers(df_energIA, "horas_alto_consumo")

Cantidad de outliers para la variable horas_alto_consumo: 0


In [ ]:
# Outliers para la variable cantidad_equipos
detectar_outliers(df_energIA, "cantidad_equipos")

Cantidad de outliers para la variable cantidad_equipos: 0


Se verificó a través de una función la posible existencia de outliers en las variables numéricas. Como se mostró en la Fase anterior, a través de los diagramas de cajas, no hay variables que presenten outliers.




## **Variables categóricas y variables booleanas**

In [ ]:
# Función outliers variables categóricas
def detectar_outliers_cat(df, variable):
  cantidad_cat = df[variable].value_counts()
  porcentaje_cat = (df[variable].value_counts(normalize=True) * 100).round(2)
  outliers_cat = pd.concat([cantidad_cat, porcentaje_cat], axis=1)
  outliers_cat.columns = ["cantidad", "porcentaje"]

  return outliers_cat

In [ ]:
# Outliers para la variable tipo_inmueble
detectar_outliers_cat(df_energIA, "tipo_inmueble")

,cantidad,porcentaje
tipo_inmueble,,
Casa,705,35.25
Departamento,596,29.80
Comercio,394,19.70
Pyme,305,15.25


In [ ]:
# Outliers para la variable calidad_aislamiento
detectar_outliers_cat(df_energIA, "calidad_aislamiento")

,cantidad,porcentaje
calidad_aislamiento,,
Media,676,33.80
Baja,437,21.85
Alta,393,19.65
Muy Baja,250,12.50
Muy Alta,244,12.20


In [ ]:
# Outliers para la variable fuente_calefaccion
detectar_outliers_cat(df_energIA, "fuente_calefaccion")

,cantidad,porcentaje
fuente_calefaccion,,
Electricidad,901,45.05
Solar,726,36.30
Otros,373,18.65


In [ ]:
# Outliers para la variable fuente_agua_caliente
detectar_outliers_cat(df_energIA, "fuente_agua_caliente")

,cantidad,porcentaje
fuente_agua_caliente,,
Electricidad,901,45.05
Solar,699,34.95
Otros,400,20.00


In [ ]:
# Outliers para la variable zona_fria
detectar_outliers_cat(df_energIA, "zona_fria")

,cantidad,porcentaje
zona_fria,,
False,1203,60.15
True,797,39.85


In [ ]:
# Outliers para la variable uso_horario_pico
detectar_outliers_cat(df_energIA, "uso_horario_pico")

,cantidad,porcentaje
uso_horario_pico,,
True,1221,61.05
False,779,38.95


In [ ]:
inmueble_error = df_energIA['tipo_inmueble'].unique().tolist()
calidad_error = df_energIA['calidad_aislamiento'].unique().tolist()
calefaccion_error = df_energIA['fuente_calefaccion'].unique().tolist()
agua_error = df_energIA['fuente_agua_caliente'].unique().tolist()
zona_error = df_energIA['zona_fria'].unique().tolist()
horario_error = df_energIA['uso_horario_pico'].unique().tolist()

print(f'''Valores únicos para la variable tipo_inmueble → {inmueble_error}
Valores únicos para la variable fuente_calefaccion → {calefaccion_error}
Valores únicos para la variable fuente_agua_caliente → {agua_error}
Valores únicos para la variable zona_fria → {zona_error}
Valores únicos para la variable uso_horario_pico → {horario_error}''')

Valores únicos para la variable tipo_inmueble → ['Departamento', 'Pyme', 'Comercio', 'Casa']
Valores únicos para la variable fuente_calefaccion → ['Solar', 'Electricidad', 'Otros']
Valores únicos para la variable fuente_agua_caliente → ['Electricidad', 'Otros', 'Solar']
Valores únicos para la variable zona_fria → [False, True]
Valores únicos para la variable uso_horario_pico → [True, False]


Para identificar posibles datos anómalos en las variables categóricas y booleanas, se tuvieron en cuenta los siguientes criterios:

1. Las cantidades y porcentajes que pudieran requerir revisión. Como referencia orientativa, se utilizó la siguiente clasificación:

| Cantidad  | Porcentaje | Categoría      |
|-----------|------------|----------------|
| ≥ 200     | ≥ 10%      | Aceptable      |
| 100 – 199 | 5% – 10%   | Poco frecuente |
| 20 – 99   | 1% – 5%    | Rara           |
| < 20      | < 1%       | Muy rara       |

2. La existencia de categorías que no estuvieran contempladas en el Diccionario de Datos.

A partir de estos criterios, no se identificaron datos anómalos en las variables analizadas. Si bien se observan diferencias en la frecuencia de algunas categorías, estas se encuentran dentro de valores esperables. Por ejemplo, en `fuente_calefaccion` se observa una diferencia aproximada del 17,65% entre la segunda y la tercera categoría. No obstante, esta diferencia no se consideraría anómala, ya que las categorías se encuentran contempladas en el diccionario de datos y sus frecuencias resultan posibles dentro del contexto.





# **Fase 9 — Análisis de correlaciones**

In [ ]:
columnas_numericas = df_energIA.select_dtypes(include=['int64', 'float64']).columns
corr = df_energIA[columnas_numericas].corr().round(3)

fig = px.imshow(
        corr,
        title="Correlación entre variables numéricas",
        text_auto=True,
        aspect="auto",
        labels=dict(
            x="",
            y="",
            color="Correlación"
        )
      )

fig.show()

La matriz muestra que no existen relaciones lineales relevantes entre las variables numéricas analizadas, debido a que los coeficientes presentan valores próximos a cero.

La mayor correlación se observa entre `metros_cuadrados` y `consumo_kwh`, con un coeficiente de 0,021, lo que indica una relación positiva prácticamente nula. En otras palabras, una mayor superficie no presenta una asociación lineal relevante con el consumo energético. Le siguen `antiguedad_vivienda` y `cantidad_equipos`, con una correlación de 0,028, y `consumo_kwh` y `horas_alto_consumo`, con -0,027.

El resto de las variables numéricas presentan correlaciones próximas a cero, por ende, no hay relaciones relevantes entre ellas.




# **Fase 10 — Distribución de la variable objetivo**

In [ ]:
# Variable categoria
orden = ["Eficiente", "Moderado", "Ineficiente"]
conteo_categoria = df_energIA["categoria"].value_counts().reindex(orden).reset_index()
conteo_categoria.columns = ["categoria", "cantidad"]

barras(conteo_categoria,
       'categoria',
       'cantidad',
       'Distribución variable categoria',
       'Categoría de eficiencia energética',
       'Cantidad',
       'Categoría')

In [ ]:
# Outliers para la variable categoria
detectar_outliers_cat(df_energIA, "categoria")

,cantidad,porcentaje
categoria,,
Moderado,1331,66.55
Eficiente,357,17.85
Ineficiente,312,15.60


La variable `categoria` presenta 3 valores: Eficiente, Moderado e Ineficiente. La clasificación se determina a partir del puntaje obtenido por cada registro: los valores superiores a 70 puntos corresponden a la categoría Eficiente; los valores entre 50 y 70 puntos, ambos inclusive, corresponden a Moderado; y los valores inferiores a 50, Ineficiente.

Como se observa en el gráfico de barras y en la tabla, la categoría predominante es Moderado, con el 66,55% de los registros. Le siguen Eficiente, con el 17,85%, e Ineficiente, con el 15,6%.

Con estos datos podemos decir que la variable objetivo no está distribuida de manera uniforme, debiendo ser considerada en las etapas posteriores de modelado, debido a un posible desbalance entre las clases.


# **Fase 11 — Relación de variables con la variable objetivo**

## **Variables categóricas y booleanas**

In [ ]:
from scipy.stats import chi2_contingency

var_cat = df_energIA.select_dtypes(exclude=['int64','float64']).columns
var_cat = var_cat.drop('hogar_id')

chi2_resultados = []

for var in var_cat:
  tabla = pd.crosstab(
      df_energIA[var],
      df_energIA['categoria']
    )

  chi2_stat, p_valores, grados_libertad, esperados = chi2_contingency(tabla)

  chi2_resultados.append({
        'variable': var,
        'metodo': 'Chi²_contingency',
        'puntaje': chi2_stat,
        'p_valor': p_valores,
        '< 0.05': np.where(p_valores < 0.05, 'Si', 'No')
    })

chi2_df = pd.DataFrame(chi2_resultados)

print(chi2_df)

               variable            metodo      puntaje       p_valor < 0.05
0         tipo_inmueble  Chi²_contingency    72.292003  1.383884e-13     Si
1             zona_fria  Chi²_contingency    48.196213  3.422357e-11     Si
2   calidad_aislamiento  Chi²_contingency   144.294197  3.031090e-27     Si
3    fuente_calefaccion  Chi²_contingency   140.602347  2.097418e-29     Si
4  fuente_agua_caliente  Chi²_contingency   174.510892  1.125078e-36     Si
5      uso_horario_pico  Chi²_contingency   217.472197  5.977401e-48     Si
6             categoria  Chi²_contingency  4000.000000  0.000000e+00     Si


Se realizó la prueba de independencia de Chi-cuadrado y p-value para evaluar la posible asociación entre las variables categóricas y booleanas y la variable `categoria`. Lo que se buscó fue identificar aquellas variables que presentan evidencia estadísticamente significativa de asociación con la variable objetivo. Para eso, se consideró como criterio de nivel de significancia de 0.05: un p-value menor a ese valor indica evidencia estadísticamente significativa de asociación.

Como se puede observar, todas las variables analizadas presentan evidencia estadísticamente signifitcativa asociada a la variable objetivo, `categoria`, dado que todos los p-value han sido menores a 0,05.

Un resultado estadísticamente significativa no implica necesariamente una relación causal, sino que indica que existe evidencia para rechazar que las variables sean necesariamente independientes entre la variable analizada y `categoria`.




## **Variables numéricas**

In [ ]:
from scipy.stats import f_oneway
from statsmodels.stats.oneway import anova_oneway

var_num = df_energIA.select_dtypes(include=['int64','float64']).columns
anova_resultados = []

for var in var_num:
  categoria_grupos = [
      grupo[var]
      for _, grupo in df_energIA.groupby('categoria')
  ]

  if var == 'metros_cuadrados':
    resultado = anova_oneway(
        categoria_grupos,
        use_var='unequal'
    )

    f_puntaje = resultado.statistic
    p_valores = resultado.pvalue
    metodo = "Welch ANOVA"

  else:
    f_puntaje, p_valores = f_oneway(*categoria_grupos)
    metodo = "ANOVA"

  anova_resultados.append({
      'variable': var,
      'metodo': metodo,
      'puntaje': f_puntaje,
      'p_valor': p_valores,
      "< 0.05": np.where(p_valores < 0.05, "Sí", "No")
  })

anova_df = pd.DataFrame(anova_resultados)

print(anova_df)

              variable       metodo     puntaje        p_valor < 0.05
0     metros_cuadrados  Welch ANOVA   25.080184   3.282824e-11     Sí
1  antiguedad_vivienda        ANOVA   11.247130   1.389116e-05     Sí
2          consumo_kwh        ANOVA  284.434278  2.027425e-109     Sí
3   horas_alto_consumo        ANOVA   18.517524   1.075482e-08     Sí
4     cantidad_equipos        ANOVA   42.412522   9.139773e-19     Sí


Para evaluar la posible asociación entre las variables numéricas y la variable `categoria`, se utilizó el método ANOVA de una vía. En el caso de la variable `metros_cuadrados`, a causa de una heterogeneidad de varianzas, se utilizó Welch ANOVA.

Se buscó identificar aquellas variables que presentan evidencia estadísticamente significativa de asociación con la variable objetivo. Para eso, se consideró como criterio de nivel de significancia de 0.05: un p-value menor a ese valor indica evidencia estadísticamente significativa de asociación.

Como se puede observar, todas las variables numéricas analizadas presentan diferencias estadísticamente significativas entre los valores de `categoria`, dado que en todos los casos en p-value es menor a 0.05.

## **Ranking de variables según significancia estadística**

In [ ]:
df_general = pd.concat([
    anova_df[["variable", "metodo", "puntaje", "p_valor", "< 0.05"]],
    chi2_df[["variable", "metodo", "puntaje", "p_valor", "< 0.05"]]
], ignore_index=True)

df_general = df_general.sort_values(
    by="p_valor",
    ascending=True
).reset_index(drop=True)

print(df_general)

                variable            metodo      puntaje        p_valor < 0.05
0              categoria  Chi²_contingency  4000.000000   0.000000e+00     Si
1            consumo_kwh             ANOVA   284.434278  2.027425e-109     Sí
2       uso_horario_pico  Chi²_contingency   217.472197   5.977401e-48     Si
3   fuente_agua_caliente  Chi²_contingency   174.510892   1.125078e-36     Si
4     fuente_calefaccion  Chi²_contingency   140.602347   2.097418e-29     Si
5    calidad_aislamiento  Chi²_contingency   144.294197   3.031090e-27     Si
6       cantidad_equipos             ANOVA    42.412522   9.139773e-19     Sí
7          tipo_inmueble  Chi²_contingency    72.292003   1.383884e-13     Si
8       metros_cuadrados       Welch ANOVA    25.080184   3.282824e-11     Sí
9              zona_fria  Chi²_contingency    48.196213   3.422357e-11     Si
10    horas_alto_consumo             ANOVA    18.517524   1.075482e-08     Sí
11   antiguedad_vivienda             ANOVA    11.247130   1.3891

El análisis evidencia que las 11 variables estudiadas presentan una relación estadísticamente significativa con la clasificación de eficiencia energética.

Entre las variables numéricas, `consumo_kwh` presenta la mayor evidencia estadística de diferencias entre las categorías, seguido por `cantidad_equipos`, `metros_cuadrados`, `horas_alto_consumo` y `antiguedad_vivienda`.

Entre las variables categóricas y booleanas, `uso_horario_pico`, `fuente_agua_caliente`, `fuente_calefaccion` presentan los menores p-values.

En conjunto, los resultados sugieren que la clasificación energética está relacionada con una combinación de factores asociados al nivel de consumo, las fuentes y hábitos de uso de la energía, el equipamiento y las características del inmueble.

# **Conclusiones finales**

## **Introducción y Objetivo de negocio**

El presente informe del Análisis Exploratorio de Datos (EDA) tiene como propósito principal responder al problema de negocio: **¿Qué factores se asocian de manera significativa con la clasificación de un usuario como eficiente, moderado o ineficiente?**. A través del análisis estadístico y de correlaciones sobre un conjunto de 2000 registros únicos de la base de datos `database_beta.json`, se han evaluado variables categóricas, booleanas y numéricas para comprender los patrones de consumo eléctrico y el entorno habitacional de los usuarios.

## **Consolidación de los hallazgos estadísticos**

Los análisis estadísticos inferenciales revelan que las 11 variables independientes del dataset poseen una relación estadísticamente significativa con la clasificación de eficiencia energética (`categoria`), utilizando un nivel de significancia estricto de `α = 0.05`.

### *Variables categóricas y booleanas*

Para las variables cualitativas, se aplicó la **prueba de independencia de Chi-cuadrado (chi²)** para determinar si la distribución de las características cualitativas varia significativamente de acuerdo con la categoría de eficiencia energética.

Los resultados muestran que todas las variables arrojaron p-values menores a 0.05, lo que aporta evidencia estadísticamente significativa para rechazar la hipótesis de independencia.

| Variable | Método Estadístico | Estadístico chi² (Puntaje) | p-value | Significación (α = 0.05) |
| :--- | :--- | :---: | :---: | :---: |
| **uso_horario_pico** | Chi²_contingency | `217.472197` | `5.977401e-48` | Sí |
| **fuente_agua_caliente** | Chi²_contingency | `174.510892` | `1.125078e-36` | Sí |
| **calidad_aislamiento** | Chi²_contingency | `144.294197` | `3.031090e-27` | Sí |
| **fuente_calefacción** | Chi²_contingency | `140.602347` | `2.097418e-29` | Sí |
| **tipo_inmueble** | Chi²_contingency | `72.292003` | `1.383884e-13` | Sí |
| **zona_fria** | Chi²_contingency | `48.196213` | `3.422357e-11` | Sí |

Las variables de comportamiento del usuario (como el consumo en horario pico) y las fuentes de suministro (calefacción y agua caliente) son los factores cualitativos con menor p-value, indicando una mayor fuerza de asociación estadística. Esto sugiere que los hábitos diarios de consumo y los sistemas energéticos principales del hogar desempeñan un rol crítico en el perfil del usuario.

### *Variables numéricas*

Se realizó un análisis de varianza para evaluar si las medias de las variables numéricas difieren entre las distintas categorías de eficiencia. Debido a la heterogeneidad de varianzas identificada en la variable `metros_cuadrados`, se aplicó **Welch ANOVA**, mientras que para el resto de las variables se utilizó el método **ANOVA de un solo factor**.

Los resultados muestran diferencias de medias estadísticamente significativas en todos los casos (`p-values < 0.05`):

| Variable | Método Estadístico | Estadístico chi² (Puntaje) | p-value | Significación (α = 0.05) |
| :--- | :--- | :---: | :---: | :---: |
| **consumo_kwh** | ANOVA | `284.434278` | `2.027425e-109` | Sí |
| **cantidad_equipos** | ANOVA | `42.412522` | `9.139773e-19` | Sí |
| **metros_cuadrados** | Welch ANOVA | `25.080184` | `3.282824e-11` | Sí |
| **horas_alto_consumo** | ANOVA | `18.517524` | `1.075482e-08` | Sí |
| **antiguedad_vivienda** | ANOVA | `11.247130` | `1.389116e-05` | Sí |

El consumo de energía es la variable cuantitativa con mayor nivel de significancia y diferencias más marcadas entre las categorías de eficiencia. Le siguen la cantidad de equipos eléctricos activos y los metros cuadrados del inmueble.

Cabe notar que la matriz de correlación lineal mostró coeficientes muy cercanos a cero entre estas variables (por ejemplo, una correlación de apenas a 0.021 entre `metros_cuadrados` y `consumo_kwh`). Esto significa que, aunque no existe una relación lineal simple a nivel global entre estas variables, sus promedios sí difieren significativamente cuando se los agrupa según el perfil de eficiencia.

## **Limitaciones críticas y contexto de los Datos**

Cualquier interpretación de estos resultados y su posterior uso deben estar condicionados a las siguientes limitaciones identificadas en el conjunto de datos analizado:

1. **Uso de datos sintéticos:** El análisis se ha desarrollado sobre un dataset simulado (`database_beta.json`), lo que significa que los registros no proceden de mediciones reales de consumo en hogares. En consecuencia, los resultados carecen de validez externa y no pueden interpretarse como representativos del comportamiento real de una población.

2. **Endogeneidad:** La variable objetivo `categoria` (Eficiente, Moderado, Ineficiente) fue preestablecida sintéticamente mediante puntuaciones basadas en las mismas variables del dataset. Por consiguiente, encontrar asociaciones estadísticamente significativas con p-values tan extremos es un resultado esperado debido al diseño circular del conjunto de datos y no un descubrimiento de relaciones independientes en la naturaleza.

3. **Ausencia de causalidad:** Se destaca que un resultado estadísticamente significativo en pruebas de Chi-cuadrado o ANOVA no implican causalidad. Indica una menor probabilidad de que las variables sean independientes, pero no que el incremento de una variable sea la causa directa del cambio en el perfil energético.

4. **Desbalance de la variable objetivo:** Se debe prestar especial atención al desbalance de la variable objetivo (con una concentración mayoritaria en el perfil Moderado de 66.55%, frente a un 17.85% Eficiente y 15.6% Ineficiente). Este comportamiento deberá ser tratado durante las siguientes fases del proyecto para evitar sesgos en el clasificador.

## **Resultado**

En conclusión, el perfil de eficiencia energética del usuario no está dictado por un único factor aislado, sino que **está asociado de manera significativa con una combinación de características físicas del inmueble, infraestructura térmica y hábitos de uso de la energía.**
